In [1]:
import pandas as pd
import glob, os
from tqdm import tqdm

In [3]:
snv_annotation = pd.read_csv("./10_12_Gregor_Trio.snv.merged/10_12_Gregor_Trio.snv.merged.hg38_multianno.txt", sep="\t")
columns = ['Chr', 'Start', 'End', 'Ref', 'Alt', 'Otherinfo12', 'Otherinfo13', 'Otherinfo14', 'Otherinfo15','REVEL_score','CADD_phred','ALL.sites.2015_08', 'CLNSIG']
snv_annotation[(snv_annotation['REVEL_score'] >='0.75') & (snv_annotation['CADD_phred'] >='20')][columns]

,Chr,Start,End,Ref,Alt,Otherinfo12,Otherinfo13,Otherinfo14,Otherinfo15,REVEL_score,CADD_phred,ALL.sites.2015_08,CLNSIG
134741,chr12,120738280,120738280,G,A,GT:GQ:DP:AD:AF:PS,"1|0:61:66:32,34:0.5152:120716821","1|0:60:53:28,25:0.4717:120716821","0|1:63:53:25,28:0.5283:120716821",0.774,28.3,0.182308,Benign/Likely_benign
188278,chr15,89325639,89325639,G,A,GT:GQ:DP:AD:AF:PS,"1|0:74:56:26,29:0.5179:89233963",./.:.:.:.:.:.,"0|1:60:69:33,36:0.5217:89233963",0.796,29.8,0.000798722,Conflicting_classifications_of_pathogenicity
221530,chr17,34959645,34959645,A,G,GT:GQ:DP:AD:AF:PS,"0|1:61:52:26,25:0.4808:34917827","0|1:19:37:23,14:0.3784:34917827",./.:.:.:.:.:.,0.858,26.3,0.290935,.
344099,chr3,14131626,14131626,T,C,GT:GQ:DP:AD:AF:PS,"1|0:57:62:31,29:0.4677:14103119",./.:.:.:.:.:.,"1|0:66:69:33,33:0.4783:14102643",0.916,27.9,.,Uncertain_significance
364197,chr3,138465387,138465387,G,A,GT:GQ:DP:AD:AF:PS,./.:.:.:.:.:.,./.:.:.:.:.:.,"1|0:64:47:22,23:0.4894:138451274",0.765,28.6,.,.


In [4]:
manifest = pd.read_csv("../../../adaptive_sampling_15.csv")

In [18]:
manifest

,run_id,sample_id,unique_id,kit_type,bed_file,proband_gender,path_to_flowcell
0,PCA100035_2025-05-20_Run1,10_12_Gregor_Trio,20250520_1236_1F_PBE50851_c0f950c7,SQK-NBD114-96,Arthrogryposis,Female,/stornext/snfs190/next-gen/ONT/PromethION/PCA1...
1,PCA100035_2025-05-20_Run1,13_15_Gregor_Trio,20250520_1237_1G_PBE41795_4209257d,SQK-NBD114-96,Microcephaly,Male,/stornext/snfs190/next-gen/ONT/PromethION/PCA1...
2,PCA100035_2025-05-20_Run1,16_18_Gregor_Trio,20250520_1237_1H_PBE51085_71daceab,SQK-NBD114-96,Microcephaly,Female,/stornext/snfs190/next-gen/ONT/PromethION/PCA1...
3,PCA100035_2025-05-20_Run1,19_21_Gregor_Trio,20250520_1237_2A_PBE56364_33e008b1,SQK-NBD114-96,Microcephaly,Female,/stornext/snfs190/next-gen/ONT/PromethION/PCA1...
4,PCA100035_2025-05-20_Run1,1_3_Gregor_Trio,20250520_1234_1A_PBE28886_15240170,SQK-NBD114-96,Epilepsy,Male,/stornext/snfs190/next-gen/ONT/PromethION/PCA1...
5,PCA100035_2025-05-20_Run1,22_24_Gregor_Trio,20250520_1238_2B_PBE57440_6d633900,SQK-NBD114-96,Neurodegeneration,Female,/stornext/snfs190/next-gen/ONT/PromethION/PCA1...
6,PCA100035_2025-05-20_Run1,4_6_Gregor_Trio,20250520_1234_1B_PBE52882_2bd2bfe2,SQK-NBD114-96,Epilepsy,Female,/stornext/snfs190/next-gen/ONT/PromethION/PCA1...
7,PCA100035_2025-05-20_Run1,7_9_Gregor_Trio,20250520_1236_1E_PBE24973_2451048f,SQK-NBD114-96,Arthrogryposis,Male,/stornext/snfs190/next-gen/ONT/PromethION/PCA1...
8,PCA100035_2025-05-20_Run2,25_27_Gregor_Trio,20250520_1247_3A_PBE56224_62f8b680,SQK-NBD114-96,Neurodegeneration,Male,/stornext/snfs190/next-gen/ONT/PromethION/PCA1...
9,PCA100035_2025-05-20_Run2,28-30_Gregor_Trio,20250520_1247_3B_PBE57257_5456e4c7,SQK-NBD114-96,Neurodegeneration,Male,/stornext/snfs190/next-gen/ONT/PromethION/PCA1...


In [20]:
# Find all annotation outputs
multiannos = sorted(glob.glob("./*/**/*.hg38_multianno.txt", recursive=True))

keep_cols = [
    'Chr', 'Start', 'End', 'Ref', 'Alt',
    'Otherinfo12', 'Otherinfo13', 'Otherinfo14', 'Otherinfo15',
    'REVEL_score', 'CADD_phred', 'ALL.sites.2015_08', 'CLNSIG'
]

out_frames = []

for path in tqdm(multiannos, desc="Processing annotation files"):
    df = pd.read_csv(path, sep="\t", dtype=str, low_memory=False)

    # Convert scores to numeric
    for c in ('REVEL_score', 'CADD_phred'):
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors='coerce')

    revel_ok = df['REVEL_score'].ge(0.75) if 'REVEL_score' in df.columns else pd.Series(False, index=df.index)
    cadd_ok  = df['CADD_phred'].ge(20)     if 'CADD_phred'  in df.columns else pd.Series(False, index=df.index)
    filt = revel_ok & cadd_ok

    cols = [c for c in keep_cols if c in df.columns]
    sub = df.loc[filt, cols].copy()

    # Extract sample name from folder (./10_12_Gregor_Trio.snv.merged/ → 10_12_Gregor_Trio.snv.merged)
    sample_folder = os.path.basename(os.path.dirname(path))

    # Match manifest sample_id (strip ".snv.merged" if needed)
    sample_id = sample_folder.replace(".snv.merged", "")
    bed_file_val = None
    match = manifest.loc[manifest['sample_id'] == sample_id]
    if not match.empty:
        bed_file_val = match['bed_file'].iloc[0]

    # Insert columns
    sub.insert(0, "bed_file", bed_file_val)
    sub.insert(0, "sample", sample_id)

    out_frames.append(sub)

# Combine all into one DataFrame
result = pd.concat(out_frames, ignore_index=True) if out_frames else pd.DataFrame(columns=["sample", "bed_file"] + keep_cols)

# Save
result.to_csv("snv_pathogenic_candidates_with_manifest.tsv", sep="\t", index=False)

Processing annotation files: 100%|██████████| 17/17 [03:31<00:00, 12.43s/it]


In [49]:
df = pd.read_csv("/users/u254106/Yilei/projects/adaptive_sampling/analysis/trio_analysis/snv_trio_annotation/4_6_Gregor_Trio.snv.merged/4_6_Gregor_Trio.snv.merged.hg38_multianno.txt", sep="\t", dtype=str, low_memory=False)
df[(df['Gene.refGeneWithVer'] == "MED27") & (df['CLNSIG'] == "Uncertain_significance")][['Chr', 'Start', 'End', 'Ref', 'Alt', 'Gene.refGeneWithVer','Otherinfo12', 'Otherinfo13', 'Otherinfo14', 'Otherinfo15','REVEL_score','CADD_phred','ALL.sites.2015_08', 'CLNSIG']]

,Chr,Start,End,Ref,Alt,Gene.refGeneWithVer,Otherinfo12,Otherinfo13,Otherinfo14,Otherinfo15,REVEL_score,CADD_phred,ALL.sites.2015_08,CLNSIG
429257,chr9,131860635,131860635,G,A,MED27,GT:GQ:DP:AD:AF:PS,"1/1:34:76:0,75:0.9868:.","0|1:54:44:25,19:0.4318:131844140","0|1:30:44:27,17:0.3864:131860635",0.199,24.1,.,Uncertain_significance


In [48]:
# 19_21
df = pd.read_csv("/users/u254106/Yilei/projects/adaptive_sampling/analysis/trio_analysis/snv_trio_annotation/19_21_Gregor_Trio.snv.merged/19_21_Gregor_Trio.snv.merged.hg38_multianno.txt", sep="\t", dtype=str, low_memory=False)
df['CADD_phred_num'] = pd.to_numeric(df['CADD_phred'], errors='coerce')
hom_alt = r'1[\/|]1'
proband_hom = df['Otherinfo13'].str.contains(hom_alt, na=False)
parent1_hom = df['Otherinfo14'].str.contains(hom_alt, na=False)
parent2_hom = df['Otherinfo15'].str.contains(hom_alt, na=False)
filtered = df[
    proband_hom &
    ~(parent1_hom | parent2_hom) &
    (df['CLNSIG'].str.contains('Pathogenic|pathogenic'))
][[
    'Chr','Start','End','Ref','Alt',
    'Otherinfo12','Otherinfo13','Otherinfo14','Otherinfo15','Gene.refGeneWithVer',
    'REVEL_score','CADD_phred','ALL.sites.2015_08','CLNSIG'
]]
filtered

,Chr,Start,End,Ref,Alt,Otherinfo12,Otherinfo13,Otherinfo14,Otherinfo15,Gene.refGeneWithVer,REVEL_score,CADD_phred,ALL.sites.2015_08,CLNSIG
421431,chr7,103989356,103989356,-,GCCGCC,GT:GQ:DP:AD:AF:PS,"1/1:33:63:1,58:0.9206:.","0|1:45:45:16,25:0.5556:103854946","0|1:43:60:28,31:0.5167:103846869",RELN,.,.,0.604233,Conflicting_classifications_of_pathogenicity


In [46]:
df = pd.read_csv("/users/u254106/Yilei/projects/adaptive_sampling/analysis/trio_analysis/snv_trio_annotation/40-42_Gregor_Trio.snv.merged/40-42_Gregor_Trio.snv.merged.hg38_multianno.txt", sep="\t", dtype=str, low_memory=False)
df['CADD_phred_num'] = pd.to_numeric(df['CADD_phred'], errors='coerce')
hom_alt = r'1[\/|]1'
proband_hom = df['Otherinfo13'].str.contains(hom_alt, na=False)
parent1_hom = df['Otherinfo14'].str.contains(hom_alt, na=False)
parent2_hom = df['Otherinfo15'].str.contains(hom_alt, na=False)
filtered = df[
    proband_hom &
    ~(parent1_hom | parent2_hom) &
    (df['CLNSIG'].str.contains('Pathogenic|pathogenic'))
][[
    'Chr','Start','End','Ref','Alt', 'Gene.refGeneWithVer',
    'Otherinfo12','Otherinfo13','Otherinfo14','Otherinfo15',
    'REVEL_score','CADD_phred','ALL.sites.2015_08','CLNSIG'
]]
filtered

,Chr,Start,End,Ref,Alt,Gene.refGeneWithVer,Otherinfo12,Otherinfo13,Otherinfo14,Otherinfo15,REVEL_score,CADD_phred,ALL.sites.2015_08,CLNSIG
68208,chr11,17430945,17430945,G,A,ABCC8,GT:GQ:DP:AD:AF:PS,"1/1:35:74:0,73:0.9865:.","0|1:68:78:39,39:0.5:17411891","1|0:59:73:37,36:0.4932:17420576",.,.,0.429912,Conflicting_classifications_of_pathogenicity
183957,chr17,70178027,70178027,-,T,KCNJ2,GT:GQ:DP:AD:AF:PS,"1/1:2:71:8,8:0.1127,.:.","2/1:0:76:13,7,9:0.1184,0.0921:.","2|0:4:53:7,6:.,0.1132:70158879",.,.,.,Conflicting_classifications_of_pathogenicity


In [62]:
import pandas as pd

df = pd.read_csv(
    "/users/u254106/Yilei/projects/adaptive_sampling/analysis/trio_analysis/snv_trio_annotation/40-42_Gregor_Trio.snv.merged/40-42_Gregor_Trio.snv.merged.hg38_multianno.txt",
    sep="\t", dtype=str, low_memory=False
)

# numeric CADD
df['CADD_phred_num'] = pd.to_numeric(df['CADD_phred'], errors='coerce')

# parents: identify ALT and missing genotypes
parent1_alt = df['Otherinfo14'].str.contains(r'0[\/|]1|1[\/|]1', na=False)
parent2_alt = df['Otherinfo15'].str.contains(r'0[\/|]1|1[\/|]1', na=False)
parent1_missing = df['Otherinfo14'].str.contains(r'\./\.', na=False)
parent2_missing = df['Otherinfo15'].str.contains(r'\./\.', na=False)

# filter for rows where parents don't carry ALT,
# and exclude both missing
filtered = df[
    # ~(parent1_alt | parent2_alt) &            
    (parent1_missing & parent2_missing)
    # &    
    # (df['CLNSIG'].str.contains('Pathogenic|pathogenic', na=False))
][[
    'Chr','Start','End','Ref','Alt','Gene.refGeneWithVer',
    'Otherinfo12','Otherinfo13','Otherinfo14','Otherinfo15',
    'REVEL_score','CADD_phred','ALL.sites.2015_08','CLNSIG'
]]

filtered


,Chr,Start,End,Ref,Alt,Gene.refGeneWithVer,Otherinfo12,Otherinfo13,Otherinfo14,Otherinfo15,REVEL_score,CADD_phred,ALL.sites.2015_08,CLNSIG
125,chr1,1362790,1362790,A,T,MXRA8,GT:GQ:DP:AD:AF:PS,"0|1:3:90:35,55:0.6111:1362790",./.:.:.:.:.:.,./.:.:.:.:.:.,.,.,0.815296,.
159,chr1,1509914,1509914,-,AA,"ATAD3B,ATAD3A",GT:GQ:DP:AD:AF,"1/2:1:63:5,5,15:0.0794,0.2381",./.:.:.:.:.,./.:.:.:.:.,.,.,.,.
160,chr1,1509914,1509914,-,A,"ATAD3B,ATAD3A",GT:GQ:DP:AD:AF,"1/2:1:63:5,5,15:0.0794,0.2381",./.:.:.:.:.,./.:.:.:.:.,.,.,.,.
165,chr1,1521299,1521301,AAA,-,ATAD3A,GT:GQ:DP:AD:AF,"1/2:0:64:6,7,4:0.1094,0.0625",./.:.:.:.:.,./.:.:.:.:.,.,.,.,.
166,chr1,1521301,1521301,-,A,ATAD3A,GT:GQ:DP:AD:AF,"1/2:0:64:6,7,4:0.1094,0.0625",./.:.:.:.:.,./.:.:.:.:.,.,.,.,.
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
454385,chrX,154563013,154563013,-,T,IKBKG,GT:GQ:DP:AD:AF:PS,"1|0:0:13:1,2:0.1538:154563013",./.:.:.:.:.:.,./.:.:.:.:.:.,.,.,.,.
454390,chrX,154565386,154565386,-,TT,IKBKG,GT:GQ:DP:AD:AF:PS,"1|0:10:9:1,2:0.2222:154563013",./.:.:.:.:.:.,./.:.:.:.:.:.,.,.,.,.
454445,chrX,155250036,155250037,AA,-,"VBP1,RAB39B",GT:GQ:DP:AD:AF:PS,"0|1:0:15:3,3:0.2:155249094",./.:.:.:.:.:.,./.:.:.:.:.:.,.,.,.,.
454446,chrX,155250067,155250067,T,-,"VBP1,RAB39B",GT:GQ:DP:AD:AF:PS,"1|0:0:15:1,1:0.0667:155249094",./.:.:.:.:.:.,./.:.:.:.:.:.,.,.,.,.


In [66]:
filtered[filtered['ALL.sites.2015_08'] == '.'].to_csv("../../small_var/40_42/all_de_novo.csv")

In [44]:
df = pd.read_csv("/users/u254106/Yilei/projects/adaptive_sampling/analysis/trio_analysis/snv_trio_annotation/31-33_Gregor_Trio.snv.merged/31-33_Gregor_Trio.snv.merged.hg38_multianno.txt", sep="\t", dtype=str, low_memory=False)
df['CADD_phred_num'] = pd.to_numeric(df['CADD_phred'], errors='coerce')
df[(df['Gene.refGeneWithVer'] == "YWHAG") & (df['CADD_phred_num'] > 20)][['Chr', 'Start', 'End', 'Ref', 'Alt', 'Gene.refGeneWithVer', 'Otherinfo12', 'Otherinfo13', 'Otherinfo14', 'Otherinfo15','REVEL_score','CADD_phred','ALL.sites.2015_08', 'CLNSIG']]

,Chr,Start,End,Ref,Alt,Gene.refGeneWithVer,Otherinfo12,Otherinfo13,Otherinfo14,Otherinfo15,REVEL_score,CADD_phred,ALL.sites.2015_08,CLNSIG
384691,chr7,76330152,76330152,G,A,YWHAG,GT:GQ:DP:AD:AF:PS,"0|1:59:87:49,38:0.4368:76317446",./.:.:.:.:.:.,./.:.:.:.:.:.,0.891,31,.,Pathogenic


In [52]:
df = pd.read_csv("/users/u254106/Yilei/projects/adaptive_sampling/analysis/trio_analysis/snv_trio_annotation/10_12_Gregor_Trio.snv.merged/10_12_Gregor_Trio.snv.merged.hg38_multianno.txt", sep="\t", dtype=str, low_memory=False)
df['CADD_phred_num'] = pd.to_numeric(df['CADD_phred'], errors='coerce')
hom_alt = r'1[\/|]1'
proband_hom = df['Otherinfo13'].str.contains(hom_alt, na=False)
parent1_hom = df['Otherinfo14'].str.contains(hom_alt, na=False)
parent2_hom = df['Otherinfo15'].str.contains(hom_alt, na=False)
filtered = df[
    proband_hom &
    ~(parent1_hom | parent2_hom) &
    (df['CADD_phred_num']>20) 
][[
    'Chr','Start','End','Ref','Alt', 'Gene.refGeneWithVer',
    'Otherinfo12','Otherinfo13','Otherinfo14','Otherinfo15',
    'REVEL_score','CADD_phred','ALL.sites.2015_08','CLNSIG'
]]
filtered

,Chr,Start,End,Ref,Alt,Gene.refGeneWithVer,Otherinfo12,Otherinfo13,Otherinfo14,Otherinfo15,REVEL_score,CADD_phred,ALL.sites.2015_08,CLNSIG
24554,chr1,102914362,102914362,G,A,COL11A1,GT:GQ:DP:AD:AF:PS,"1/1:35:41:0,41:1:.","1|0:57:43:25,17:0.3953:102866753","0|1:54:51:30,21:0.4118:102866753",0.604,24.7,0.496605,Benign
42334,chr1,214641390,214641390,T,G,CENPF,GT:GQ:DP:AD:AF:PS,"1/1:31:68:0,68:1:.","0|1:53:39:25,14:0.359:214593203","1|0:69:50:28,22:0.44:214595978",0.054,20.3,0.105232,Benign
93767,chr11,44309959,44309959,C,G,ALX4,GT:GQ:DP:AD:AF:PS,"1/1:32:58:0,58:1:.","1|0:57:49:26,22:0.449:44262893","0|1:62:68:33,35:0.5147:44251874",0.494,26.9,0.472644,Benign
100881,chr11,85655627,85655627,G,A,TMEM126A,GT:GQ:DP:AD:AF:PS,"1/1:30:37:0,37:1:.","0|1:61:49:22,27:0.551:85655627","0|1:60:40:17,23:0.575:85652050",0.497,22.3,0.00299521,Conflicting_classifications_of_pathogenicity
101593,chr11,95923947,95923947,T,G,MTMR2,GT:GQ:DP:AD:AF:PS,"1/1:33:39:0,38:0.9744:.","0|1:66:36:18,18:0.5:95812006","1|0:64:41:19,22:0.5366:95811944",0.320,21.5,0.212859,Benign
106675,chr11,118893788,118893788,A,C,CXCR5,GT:GQ:DP:AD:AF:PS,"1/1:31:45:0,43:0.9556:.","0|1:69:54:19,31:0.5741:118883940","0|1:54:49:31,17:0.3469:118886482",0.136,22.0,.,.
136529,chr12,132142314,132142314,C,T,DDX51,GT:GQ:DP:AD:AF:PS,"1/1:31:59:0,59:1:.","1|0:63:54:26,28:0.5185:132132006","1|0:64:60:27,33:0.55:132128694",0.486,29.1,0.00439297,.
138005,chr13,23333770,23333770,A,G,SACS,GT:GQ:DP:AD:AF:PS,"1/1:60:23:0,23:1:.","1|0:38:29:18,11:0.3793:23291217","0|1:51:20:10,10:0.5:23172420",0.339,23.1,0.258986,Benign/Likely_benign
138143,chr13,24321667,24321667,G,A,C1QTNF9,GT:GQ:DP:AD:AF:PS,"1/1:31:59:0,56:0.9492:.","1|0:62:39:20,19:0.4872:24305174","0|1:53:39:24,15:0.3846:24305174",0.717,26.4,0.0792732,.
140077,chr13,28427872,28427872,G,A,FLT1,GT:GQ:DP:AD:AF:PS,"1/1:33:61:0,59:0.9672:.","0|1:76:51:27,24:0.4706:28417765","0|1:73:54:28,26:0.4815:28417765",0.318,23.2,.,.


In [54]:
import pandas as pd

df = pd.read_csv(
    "/users/u254106/Yilei/projects/adaptive_sampling/analysis/trio_analysis/snv_trio_annotation/37-39_Gregor_Trio.snv.merged/37-39_Gregor_Trio.snv.merged.hg38_multianno.txt",
    sep="\t", dtype=str, low_memory=False
)

# numeric CADD
df['CADD_phred_num'] = pd.to_numeric(df.get('CADD_phred'), errors='coerce')

# homozygous alt (phased or unphased)
hom_alt = r'1[\/|]1'
proband_hom = df.get('Otherinfo13', '').str.contains(hom_alt, na=False)
parent1_hom = df.get('Otherinfo14', '').str.contains(hom_alt, na=False)
parent2_hom = df.get('Otherinfo15', '').str.contains(hom_alt, na=False)

# pathogenic flag from CLNSIG (case-insensitive)
pathogenic = df.get('CLNSIG', '').str.contains(r'pathogenic', case=False, na=False)

filtered = df[
    proband_hom &
    ~(parent1_hom | parent2_hom) &
    ((df['CADD_phred_num'] > 20) | pathogenic)
][[
    c for c in [
        'Chr','Start','End','Ref','Alt','Gene.refGeneWithVer',
        'Otherinfo12','Otherinfo13','Otherinfo14','Otherinfo15',
        'REVEL_score','CADD_phred','ALL.sites.2015_08','CLNSIG'
    ] if c in df.columns
]]

filtered


,Chr,Start,End,Ref,Alt,Gene.refGeneWithVer,Otherinfo12,Otherinfo13,Otherinfo14,Otherinfo15,REVEL_score,CADD_phred,ALL.sites.2015_08,CLNSIG
589,chr1,1512312,1512312,G,A,ATAD3A,GT:GQ:DP:AD:AF:PS,"1/1:33:115:0,114:0.9913:.","0|1:62:64:30,34:0.5312:1472751","0|1:66:77:43,34:0.4416:1489332",0.327,22.4,0.179712,.
10502,chr1,36085057,36085057,C,T,TEKT2,GT:GQ:DP:AD:AF:PS,"1/1:29:73:0,73:1:.","0|1:62:46:22,24:0.5217:36079782","0|1:57:54:24,29:0.537:36079782",0.162,25.0,0.0734824,Benign
24943,chr1,100190140,100190141,AA,-,DBT,GT:GQ:DP:AD:AF:PS,"1/1:110:76:1,75:0.9868:.","0|1:52:47:24,23:0.4894:100190139","0|1:62:70:34,36:0.5143:100179499",.,.,.,Conflicting_classifications_of_pathogenicity
58121,chr10,69408971,69408971,C,T,TACR2,GT:GQ:DP:AD:AF:PS,"1/1:33:64:2,62:0.9688:.","0|1:52:31:16,12:0.3871:69375392","0|1:66:41:18,22:0.5366:69263428",0.349,22.2,0.126597,.
83791,chr11,117351931,117351931,-,A,CEP164,GT:GQ:DP:AD:AF:PS,"1/1:19:91:32,17:0.1868:.","0|1:5:48:14,8:0.1667:117341455",./.:.:.:.:.:.,.,.,.,Conflicting_classifications_of_pathogenicity
98268,chr12,89351700,89351700,C,A,DUSP6,GT:GQ:DP:AD:AF:PS,"1/1:33:87:0,87:1:.","1|0:51:56:33,23:0.4107:89342647","0|1:60:48:17,31:0.6458:89342121",0.099,23.4,0.466254,Benign
157762,chr16,15727006,15727006,C,T,MYH11,GT:GQ:DP:AD:AF:PS,"1/1:33:99:0,98:0.9899:.","1|0:63:49:25,24:0.4898:15633602","0|1:66:53:26,27:0.5094:15634091",0.247,23.7,0.203874,Benign
181796,chr17,42562786,42562786,C,A,COASY,GT:GQ:DP:AD:AF:PS,"1/1:31:101:1,99:0.9802:.","1|0:59:49:19,30:0.6122:42526354","1|0:59:56:23,33:0.5893:42526354",0.064,23.3,0.506989,Benign
184232,chr17,75522122,75522122,G,C,TSEN54,GT:GQ:DP:AD:AF:PS,"1/1:32:113:1,111:0.9823:.","0|1:62:56:33,23:0.4107:75506555","0|1:70:49:25,24:0.4898:75506037",0.118,24.0,0.444688,Benign
252790,chr21,46125854,46125854,G,A,COL6A2,GT:GQ:DP:AD:AF:PS,"1/1:33:89:1,88:0.9888:.","0|1:58:79:43,34:0.4304:46088081","0|1:61:66:26,36:0.5455:46088081",0.277,25.4,0.39397,Benign


In [55]:
import pandas as pd

df = pd.read_csv(
    "/users/u254106/Yilei/projects/adaptive_sampling/analysis/trio_analysis/snv_trio_annotation/28-30_Gregor_Trio.snv.merged/28-30_Gregor_Trio.snv.merged.hg38_multianno.txt",
    sep="\t", dtype=str, low_memory=False
)

# numeric CADD
df['CADD_phred_num'] = pd.to_numeric(df.get('CADD_phred'), errors='coerce')

# homozygous alt (phased or unphased)
hom_alt = r'1[\/|]1'
proband_hom = df.get('Otherinfo13', '').str.contains(hom_alt, na=False)
parent1_hom = df.get('Otherinfo14', '').str.contains(hom_alt, na=False)
parent2_hom = df.get('Otherinfo15', '').str.contains(hom_alt, na=False)

# pathogenic flag from CLNSIG (case-insensitive)
pathogenic = df.get('CLNSIG', '').str.contains(r'pathogenic', case=False, na=False)

filtered = df[
    proband_hom &
    ~(parent1_hom | parent2_hom) &
    ((df['CADD_phred_num'] > 20) | pathogenic)
][[
    c for c in [
        'Chr','Start','End','Ref','Alt','Gene.refGeneWithVer',
        'Otherinfo12','Otherinfo13','Otherinfo14','Otherinfo15',
        'REVEL_score','CADD_phred','ALL.sites.2015_08','CLNSIG'
    ] if c in df.columns
]]

filtered


,Chr,Start,End,Ref,Alt,Gene.refGeneWithVer,Otherinfo12,Otherinfo13,Otherinfo14,Otherinfo15,REVEL_score,CADD_phred,ALL.sites.2015_08,CLNSIG
109106,chr13,37009694,37009694,G,A,SUPT20H,GT:GQ:DP:AD:AF:PS,"1/1:32:55:0,54:0.9818:.","1|0:62:34:13,21:0.6176:36989654","1|0:53:20:5,15:0.75:37005862",0.026,21.4,0.413139,.
125298,chr14,88385822,88385822,G,A,SPATA7,GT:GQ:DP:AD:AF:PS,"1/1:30:53:0,53:1:.","0|1:67:32:16,16:0.5:88377721","0|1:58:28:16,12:0.4286:88380876",0.055,20.5,0.189896,Benign
144909,chr15,89316763,89316763,C,A,POLG,GT:GQ:DP:AD:AF:PS,"1/1:30:56:0,56:1:.","0|1:51:34:19,15:0.4412:89295741","0|1:49:28:14,14:0.5:89295741",0.358,20.8,0.0269569,Benign/Likely_benign
161516,chr16,15725134,15725134,-,A,NDE1,GT:GQ:DP:AD:AF:PS,"1/1:3:68:25,9:0.1324,.:.","1|0:9:48:14,6:0.125,.:15634417","0|2:8:43:16,3:.,0.0698:15633602",.,.,.,Conflicting_classifications_of_pathogenicity
182484,chr17,21702902,21702902,G,A,KCNJ18,GT:GQ:DP:AD:AF:PS,"1/1:43:59:8,51:0.8644:.","0|1:60:65:33,32:0.4923:21682930","0|1:47:28:16,12:0.4286:21682507",.,23.1,.,Likely_benign
191695,chr18,10681714,10681714,C,T,PIEZO2,GT:GQ:DP:AD:AF:PS,"1/1:30:63:1,61:0.9683:.","1|0:66:43:25,18:0.4186:10660862","0|1:68:27:10,17:0.6296:10661942",0.083,20.8,0.115615,Benign
212426,chr19,43553422,43553422,G,A,XRCC1,GT:GQ:DP:AD:AF:PS,"1/1:36:102:1,101:0.9902:.","1|0:51:49:17,32:0.6531:43496905","0|1:54:36:20,14:0.3889:43528017",0.198,24.3,0.123802,Benign
213966,chr19,49876651,49876651,G,A,AKT1S1,GT:GQ:DP:AD:AF:PS,"1/1:30:111:1,104:0.9369:.","0|1:59:53:25,27:0.5094:49876651","0|1:55:44:18,25:0.5682:49859918",0.162,22.4,0.000399361,.
242436,chr2,218282080,218282080,G,A,TMBIM1,GT:GQ:DP:AD:AF:PS,"1/1:33:65:0,64:0.9846:.","1|0:62:58:28,29:0.5:218260198","1|0:62:38:17,21:0.5526:218260198",0.256,24.2,0.449481,.
295068,chr3,129565975,129565975,A,C,PLXND1,GT:GQ:DP:AD:AF:PS,"1/1:31:63:0,63:1:.","0|1:63:47:21,26:0.5532:129545429","1|0:58:34:16,18:0.5294:129563023",0.184,22.4,0.176717,Benign


In [58]:
# list(df.columns)